In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from tqdm import tqdm
from scipy.signal import savgol_filter
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

from lightgbm import LGBMRegressor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from scipy.spatial.distance import euclidean

In [2]:
BASE_PATH = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH = os.path.join(BASE_PATH, "test")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

WINDOW_SIZE = 256
STRIDE = 64
BATCH_SIZE = 32
EPOCHS = 12

geo_targets = [
    'ANCC',
    'ASTNU',
    'ASTNL',
    'EGFDU',
    'EGFDL',
    'BUDA'
]

def create_features(df):

    df = df.copy()
    df['dx'] = df['X'].diff().fillna(0)
    df['dy'] = df['Y'].diff().fillna(0)
    df['dz'] = df['Z'].diff().fillna(0)
    df['dmd'] = df['MD'].diff().fillna(1)
    df['horizontal_dist'] = np.sqrt(df['dx']**2 + df['dy']**2)
    df['trajectory_dist'] = np.sqrt(
        df['dx']**2 +
        df['dy']**2 +
        df['dz']**2
    )
    df['inclination'] = np.arctan2(
        np.abs(df['dz']),
        df['horizontal_dist'] + 1e-6
    )
    df['curvature'] = np.sqrt(
        df['dx'].diff().fillna(0)**2 +
        df['dy'].diff().fillna(0)**2 +
        df['dz'].diff().fillna(0)**2
    )
    df['relative_md'] = df['MD'] / (df['MD'].max() + 1e-6)
    for w in [5, 15, 30]:
        df[f'GR_mean_{w}'] = (
            df['GR']
            .rolling(w)
            .mean()
            .bfill()
        )

        df[f'GR_std_{w}'] = (
            df['GR']
            .rolling(w)
            .std()
            .fillna(0)
        )

        df[f'GR_min_{w}'] = (
            df['GR']
            .rolling(w)
            .min()
            .bfill()
        )

        df[f'GR_max_{w}'] = (
            df['GR']
            .rolling(w)
            .max()
            .bfill()
        )

    df['GR_gradient'] = df['GR'].diff().fillna(0)
    df['Z_gradient'] = df['Z'].diff().fillna(0)

    df['TVT_input_filled'] = (
        df['TVT_input']
        .ffill()
        .bfill()
    )

    df['TVT_input_grad'] = (
        df['TVT_input_filled']
        .diff()
        .fillna(0)
    )

    return df

def load_typewell(path):
    tw = pd.read_csv(path)
    tw = tw.sort_values('TVT').reset_index(drop=True)
    return tw

In [3]:
def dtw_path(x, y):

    n = len(x)
    m = len(y)

    dtw = np.full((n + 1, m + 1), np.inf)

    dtw[0, 0] = 0

    for i in range(1, n + 1):

        for j in range(1, m + 1):

            cost = abs(x[i - 1] - y[j - 1])

            dtw[i, j] = cost + min(
                dtw[i - 1, j],
                dtw[i, j - 1],
                dtw[i - 1, j - 1]
            )

    i = n
    j = m

    path = []

    while i > 0 and j > 0:

        path.append((i - 1, j - 1))

        steps = [
            dtw[i - 1, j],
            dtw[i, j - 1],
            dtw[i - 1, j - 1]
        ]

        argmin = np.argmin(steps)

        if argmin == 0:
            i -= 1
        elif argmin == 1:
            j -= 1
        else:
            i -= 1
            j -= 1

    path.reverse()

    return dtw[n, m], path

def compute_dtw_features(horizontal_df, typewell_df):

    h_gr = horizontal_df['GR'].fillna(0).values
    t_gr = typewell_df['GR'].fillna(0).values

    distance, path = dtw_path(h_gr, t_gr)

    dtw_tvt = np.zeros(len(horizontal_df))
    dtw_conf = np.zeros(len(horizontal_df))

    mapping = {}

    for i, j in path:

        if i not in mapping:
            mapping[i] = []

        mapping[i].append(j)

    for i in range(len(horizontal_df)):

        if i in mapping:

            js = mapping[i]

            mapped_idx = int(np.mean(js))

            mapped_idx = np.clip(
                mapped_idx,
                0,
                len(typewell_df) - 1
            )

            dtw_tvt[i] = (
                typewell_df
                .iloc[mapped_idx]['TVT']
            )

            dtw_conf[i] = len(js)

        else:

            dtw_tvt[i] = np.nan
            dtw_conf[i] = 0

    horizontal_df['dtw_tvt'] = (
        pd.Series(dtw_tvt)
        .interpolate()
        .bfill()
        .ffill()
    )

    horizontal_df['dtw_confidence'] = dtw_conf
    horizontal_df['dtw_distance_global'] = distance

    return horizontal_df

def load_data(path, is_train=True):
    dfs = []
    for file in tqdm(os.listdir(path)):
        if 'horizontal_well.csv' not in file:
            continue
        well_id = file.split('__')[0]
        horizontal_path = os.path.join(path, file)
        typewell_path = os.path.join(path, f'{well_id}__typewell.csv')
        h_df = pd.read_csv(horizontal_path)
        t_df = load_typewell(typewell_path)
        h_df['well_id'] = well_id
        h_df['row_id'] = np.arange(len(h_df))
        h_df = create_features(h_df)
        h_df = compute_dtw_features(h_df, t_df)
        dfs.append(h_df)
    return pd.concat(dfs, ignore_index=True)

In [4]:
train_df = load_data(TRAIN_PATH, is_train=True)
test_df = load_data(TEST_PATH, is_train=False)
print(train_df.shape)
print(test_df.shape)

100%|██████████| 6/6 [00:43<00:00,  7.30s/it]

(5092255, 43)
(19221, 36)


In [5]:
base_features = [
    c for c in train_df.columns
    if c not in geo_targets + ['TVT', 'TVT_input', 'well_id']
]
geo_models = {}
gkf = GroupKFold(n_splits=5)
for target in geo_targets:
    print(f'\nTraining geological surface: {target}')
    X = train_df[base_features]
    y = train_df[target]
    oof = np.zeros(len(train_df))
    for fold, (tr_idx, va_idx) in enumerate(
        gkf.split(X, y, train_df['well_id'])
    ):
        model = LGBMRegressor(
            n_estimators=2500,
            learning_rate=0.02,
            num_leaves=128,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='regression',
            random_state=SEED
        )
        model.fit(
            X.iloc[tr_idx],
            y.iloc[tr_idx],
            eval_set=[(
                X.iloc[va_idx],
                y.iloc[va_idx]
            )],
            eval_metric='rmse'
        )
        oof[va_idx] = model.predict(X.iloc[va_idx])
    geo_models[target] = model
for target in geo_targets:
    test_df[target] = geo_models[target].predict(test_df[base_features])

def add_geo_features(df):
    df = df.copy()
    for col in geo_targets:
        df[f'dist_to_{col}'] = df['Z'] - df[col]
    df['relative_egfdl_buda'] = (
        (df['Z'] - df['EGFDL']) /
        (df['BUDA'] - df['EGFDL'] + 1e-6)
    )

    df['relative_astnu_astnl'] = (
        (df['Z'] - df['ASTNU']) /
        (df['ASTNL'] - df['ASTNU'] + 1e-6)
    )

    return df

train_df = add_geo_features(train_df)
test_df = add_geo_features(test_df)
train_df['TVT_delta'] = (
    train_df
    .groupby('well_id')['TVT']
    .diff()
    .fillna(0)
)
DROP_COLS = [
    'TVT',
    'TVT_input',
    'well_id'
]

features = [
    c for c in train_df.columns
    if c not in DROP_COLS
]
print('\nTraining LightGBM...')
X = train_df[features]
y = train_df['TVT']
oof_lgb = np.zeros(len(train_df))
test_preds_lgb = np.zeros(len(test_df))
for fold, (tr_idx, va_idx) in enumerate(
    gkf.split(X, y, train_df['well_id'])
):
    print(f'Fold {fold}')
    model = LGBMRegressor(
        n_estimators=4000,
        learning_rate=0.01,
        num_leaves=256,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='regression',
        random_state=SEED
    )
    model.fit(
        X.iloc[tr_idx],
        y.iloc[tr_idx],
        eval_set=[(
            X.iloc[va_idx],
            y.iloc[va_idx]
        )],
        eval_metric='rmse'
    )
    val_pred = model.predict(X.iloc[va_idx])
    oof_lgb[va_idx] = val_pred
    test_preds_lgb += (
        model.predict(test_df[features]) / 5
    )
rmse_lgb = np.sqrt(mean_squared_error(y, oof_lgb))
print(f'LightGBM RMSE: {rmse_lgb}')


Training geological surface: ANCC
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.295048 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8025
[LightGBM] [Info] Number of data points in the train set: 4075197, number of used features: 34
[LightGBM] [Info] Start training from score -8925.260387
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.313028 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8018
[LightGBM] [Info] Number of data points in the train set: 4075198, number of used features: 34
[LightGBM] [Info] Start training from score -8960.663479
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.294692 seconds.
You can set `force_row_wise=true` to 

KeyError: "['TVT_delta'] not in index"

In [ ]:
seq_features = [
    'GR',
    'Z',
    'X',
    'Y',
    'MD',
    'dx',
    'dy',
    'dz',
    'GR_gradient',
    'relative_md',
    'dtw_tvt',
    'dtw_confidence',
    'relative_egfdl_buda',
    'relative_astnu_astnl'
]

scaler = StandardScaler()

train_df[seq_features] = scaler.fit_transform(
    train_df[seq_features]
)

test_df[seq_features] = scaler.transform(
    test_df[seq_features]
)

def create_windows(df, features, target=None):

    Xs = []
    ys = []
    meta = []

    for well_id in df['well_id'].unique():

        wdf = (
            df[df['well_id'] == well_id]
            .sort_values('MD')
            .reset_index(drop=True)
        )

        X = wdf[features].values

        if target is not None:
            y = wdf[target].values

        for start in range(0, len(wdf)-WINDOW_SIZE, STRIDE):

            end = start + WINDOW_SIZE

            Xs.append(X[start:end])

            if target is not None:
                ys.append(y[start:end])

            meta.append((well_id, start, end))

    Xs = np.array(Xs, dtype=np.float32)

    if target is not None:
        ys = np.array(ys, dtype=np.float32)
        return Xs, ys, meta

    return Xs, meta

X_seq, y_seq, meta_seq = create_windows(
    train_df,
    seq_features,
    target='TVT'
)

X_test_seq, test_meta = create_windows(
    test_df,
    seq_features,
    target=None
)

print(X_seq.shape)

class SeqDataset(Dataset):

    def __init__(self, X, y=None):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        if self.y is None:
            return torch.tensor(self.X[idx])

        return (
            torch.tensor(self.X[idx]),
            torch.tensor(self.y[idx])
        )

class TemporalCNN(nn.Module):

    def __init__(self, in_channels):

        super().__init__()

        self.net = nn.Sequential(

            nn.Conv1d(in_channels, 64, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.BatchNorm1d(64),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(128),

            nn.Conv1d(128, 128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(128),

            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv1d(64, 1, kernel_size=1)
        )

    def forward(self, x):

        x = x.permute(0, 2, 1)

        x = self.net(x)

        x = x.squeeze(1)

        return x

In [ ]:
train_dataset = SeqDataset(X_seq, y_seq)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

model = TemporalCNN(len(seq_features)).to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print('\nTraining Temporal CNN...')

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        preds = model(xb)

        loss = criterion(preds, yb)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}')

In [ ]:
model.eval()

cnn_preds = {}
cnn_counts = {}

for well_id in test_df['well_id'].unique():

    mask = test_df['well_id'] == well_id

    cnn_preds[well_id] = np.zeros(mask.sum())
    cnn_counts[well_id] = np.zeros(mask.sum())

with torch.no_grad():

    for i in range(len(X_test_seq)):

        x = torch.tensor(X_test_seq[i:i+1]).to(DEVICE)

        pred = model(x).cpu().numpy()[0]

        well_id, start, end = test_meta[i]

        cnn_preds[well_id][start:end] += pred
        cnn_counts[well_id][start:end] += 1

In [ ]:
final_cnn_preds = []

for well_id in test_df['well_id'].unique():

    p = cnn_preds[well_id]
    c = cnn_counts[well_id]

    c[c == 0] = 1

    pred = p / c

    final_cnn_preds.extend(pred)

test_df['cnn_pred'] = final_cnn_preds

test_df['dtw_pred'] = test_df['dtw_tvt']

test_df['tvt_pred'] = (
    0.45 * test_preds_lgb +
    0.40 * test_df['cnn_pred'] +
    0.15 * test_df['dtw_pred']
)

In [ ]:
smoothed = []

for well_id in test_df['well_id'].unique():

    wdf = (
        test_df[test_df['well_id'] == well_id]
        .sort_values('MD')
    )

    pred = wdf['tvt_pred'].values

    if len(pred) > 25:
        pred = savgol_filter(pred, 21, 3)

    smoothed.extend(pred)

test_df['tvt_pred'] = smoothed

In [ ]:
submission_df = test_df[
    test_df['TVT_input'].isna()
].copy()

submission_df['id'] = (
    submission_df['well_id'] +
    '_' +
    submission_df['row_id'].astype(str)
)

submission = submission_df[
    ['id', 'tvt_pred']
].rename(columns={
    'tvt_pred': 'tvt'
})

sample = pd.read_csv(
    os.path.join(BASE_PATH, 'sample_submission.csv')
)

submission = sample[['id']].merge(
    submission,
    on='id',
    how='left'
)

submission.to_csv('submission.csv', index=False)

print('\nSubmission created!')
print(submission.head())
print('\nMissing values:')
print(submission.isna().sum())
print('\nRows:', len(submission))
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(rmse)